In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
from torch import nn
from tensordict import TensorDict

# 将项目根目录添加到 Python 路径
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    print(f"Adding project root to Python path: {project_root}")
    sys.path.insert(0, project_root)

# 导入必要模块
from implement.generator import MTHFVRPGenerator
from implement.environment import MTHFVRPEnv
from implement.model import TransformerModel

In [ ]:
def demo_model_env_interaction():
    # 1. 准备配置
    NODE_NUM = 20
    VEHICLE_NUM = 5
    BATCH_SIZE = 2  # 演示 Batch 操作
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 2. 初始化生成器和环境
    # disable check_solution for speed in demo
    generator = MTHFVRPGenerator(num_loc=NODE_NUM, vehicle_num=VEHICLE_NUM, variant_preset="hfcvrp")
    env = MTHFVRPEnv(generator=generator, batch_size=[BATCH_SIZE], device=device)

    # 3. 初始化模型
    # 我们需要先跑一次 reset 获取特征维度，以便初始化模型参数
    tmp_state = env.reset()
    input_features = env.get_global_features(tmp_state)
    
    # 自动推断特征维度
    state_feature_dims = {}
    for key, feature in input_features["state_feature"].items():
        state_feature_dims[key] = feature.shape[-1]
    
    # 获取 current feature 维度 (减2是因为模型内部会把 current node/vehicle idx 拿去做 embedding，不算在数值特征里)
    tmp_curr_feat, _ = env.get_current_feature_and_mask(tmp_state)
    state_feature_dims['current_feature'] = tmp_curr_feat.shape[-1] - 2 
    
    print("\n特征维度推断:", state_feature_dims)

    # 初始化 Transformer 模型
    model = TransformerModel(
        hidden_size=128,
        n_head=8,
        encoder_num_layers=3, # 演示用，层数少点
        state_feature_dims=state_feature_dims
    ).to(device)
    
    model.eval() # 推理模式

    print("\n=== 开始交互流程 ===")
    
    # A. Reset 环境
    state = env.reset()
    
    # B. Model Encoding (只做一次!)
    # 获取全局特征
    global_features_td = env.get_global_features(state)
    # 编码，此时模型内部会缓存 KV Cache (self.cache_*)
    model.feature(global_features_td)
    print("Encoder 编码完成，KV Cache 已准备就绪。")
    
    # C. Rollout Loop
    done = False
    step = 0
    total_rewards = torch.zeros(BATCH_SIZE, device=device)
    
    while not done:
        # 1. 获取当前步的 Context (B, D_curr) 和 Mask (B, ActionSpace)
        current_features, illegal_mask = env.get_current_feature_and_mask(state)
        
        # 2. Model Decoding
        # 输入: 当前特征 + Mask
        # 输出: 动作 Logits (未归一化的概率)
        with torch.no_grad():
            action_logits = model.policy(current_features, illegal_mask)
        
        # 3. 动作选择 (Sampling)
        action_probs = torch.softmax(action_logits, dim=-1)
        dist = torch.distributions.Categorical(action_probs)
        action = dist.sample()
        
        # 4. Environment Step
        state, reward, is_done_tensor = env.step(state, action)
        
        # 累加奖励 (注意 masked reward)
        total_rewards += reward.squeeze(-1)
        
        done = is_done_tensor.all().item()
        step += 1
        
        # 打印部分动作 (只看 Batch 0)
        if step <= 10: # 防止刷屏
            print(f"Step {step}: User 0 Action {action[0].item()} | Reward {reward[0].item():.2f}")
            
    print(f"\nEpisode 结束，共 {step} 步。")
    print(f"Final Rewards: {total_rewards.cpu().numpy()}")

# 运行演示
demo_model_env_interaction()